# 10 – Thêm seed cho M2, M3, XLM-R (W7-02)

**Người chạy: Quân** (Drive của Quân có checkpoint `checkpoints/M5_S1_seed42` cần cho XLM-R S3). Tổng khoảng **2,5 giờ** GPU T4,
chia nhiều lần được: XLM-R tự bỏ qua lần đã có kết quả; M2, M3 chỉ chạy đúng các seed ghi trong lệnh.

Mục đích: xóa hạn chế "chỉ chạy 1 seed" ở mục 5.6 báo cáo. Chạy thêm seed 123 và 2024 cho:
- M2, M3 kịch bản S1 (M2 vài phút, M3 khoảng 8 phút mỗi seed);
- XLM-R (M5) kịch bản S2 (khoảng 12 phút mỗi seed), S3 (khoảng 12 phút mỗi seed, học tiếp từ checkpoint S1 seed 42 như PhoBERT S3)
  và S1 (khoảng 35 phút mỗi seed).

**Bắt buộc dùng GPU:** Runtime → Change runtime type → **T4 GPU** → Save.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results', 'checkpoints']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

## Kiểm tra GPU và vector từ fastText (M2, M3 cần)

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU, bật T4 GPU rồi chạy lại từ ô đầu')
if not os.path.exists('data/processed/emb_fasttext.npz'):
    !python scripts/prepare_embeddings.py
else:
    print('Đã có emb_fasttext.npz')

## Bước 1: M2 và M3 kịch bản S1, seed 123 và 2024 (khoảng 20 phút)

In [ ]:
!python scripts/run_m2.py --scenario S1 --seeds 123 2024
!python scripts/run_m3.py --scenario S1 --seeds 123 2024

## Bước 2: XLM-R kịch bản S2, seed 123 và 2024 (khoảng 25 phút, cấu hình B như PhoBERT S2)

In [ ]:
!python scripts/run_transformer.py --model M5 --scenario S2 --config B --seeds 123 2024

## Bước 3: XLM-R kịch bản S3, seed 123 và 2024 (khoảng 25 phút)
Học tiếp từ `checkpoints/M5_S1_seed42` (giống PhoBERT S3: mọi seed bắt đầu từ cùng checkpoint S1 seed 42).

In [ ]:
import os
print('Có checkpoint XLM-R S1 seed 42:', os.path.isdir('checkpoints/M5_S1_seed42'))
!python scripts/run_transformer.py --model M5 --scenario S3 --config B --seeds 123 2024

## Bước 4: XLM-R kịch bản S1, seed 123 và 2024 (khoảng 70 phút, cấu hình A)

In [ ]:
!python scripts/run_transformer.py --model M5 --scenario S1 --config A --seeds 123 2024

## Bước 5: kiểm tra

In [ ]:
!python scripts/check_w7.py --task 02

**Đạt khi:** dòng cuối ghi `Đủ 26 file kết quả. Xong W7-02.`

## Lưu và nộp
Kết quả tự nằm trên Drive `NLI/results/`. Sheet: W7-02 → **Đang review**, nhắn Claude đồng bộ và phân tích.